# 03 — Root Cause Exploration

**Project:** AdIntel AI  
**Purpose:** identify which dimensions most likely explain quality decline.

This notebook uses CSV files only and focuses on portfolio-ready RCA storytelling:

1. what happened,
2. where it happened,
3. why it likely happened,
4. why data quality is not the main cause,
5. recommended business actions.


## 1. Setup

In [ ]:

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

def find_project_root(start: Path | None = None) -> Path:
    """
    Find project root by walking upward until data/synthetic exists.
    This makes the notebook runnable from:
    - project root
    - notebooks/
    - VS Code interactive working directory
    """
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]

    for candidate in candidates:
        if (candidate / "data" / "synthetic").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root. Please run this notebook from the project root "
        "or from a folder inside the project. Expected folder: data/synthetic/"
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "synthetic"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir    : {DATA_DIR}")

def read_csv_required(filename: str, parse_dates: list[str] | None = None) -> pd.DataFrame:
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

    df = pd.read_csv(path)

    for col in parse_dates or []:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    return df

def safe_divide(numerator, denominator):
    denominator = pd.Series(denominator).replace(0, np.nan)
    return numerator / denominator

def pct_change(post, base):
    if pd.isna(base) or base == 0:
        return np.nan
    return post / base - 1

def fmt_pct(x):
    return "n/a" if pd.isna(x) else f"{x:.2%}"

def fmt_num(x):
    if pd.isna(x):
        return "n/a"
    if abs(x) >= 1_000_000:
        return f"{x/1_000_000:.2f}M"
    if abs(x) >= 1_000:
        return f"{x/1_000:.2f}K"
    return f"{x:,.2f}"

def add_period_by_date(df: pd.DataFrame, date_col: str = "date") -> tuple[pd.DataFrame, pd.Timestamp]:
    """
    Split data dynamically:
    - first half of available date range = baseline
    - second half of available date range = decline
    """
    if date_col not in df.columns:
        raise KeyError(f"Column `{date_col}` not found.")

    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col], errors="coerce")

    min_date = out[date_col].min()
    max_date = out[date_col].max()

    if pd.isna(min_date) or pd.isna(max_date):
        raise ValueError(f"Column `{date_col}` does not contain valid dates.")

    midpoint = min_date + (max_date - min_date) / 2
    out["period"] = np.where(out[date_col] <= midpoint, "baseline", "decline")

    return out, midpoint

def require_columns(df: pd.DataFrame, cols: list[str], table_name: str = "dataframe"):
    missing = [col for col in cols if col not in df.columns]
    if missing:
        raise KeyError(f"{table_name} is missing required columns: {missing}")

def weighted_rate(df: pd.DataFrame, numerator_col: str, denominator_col: str) -> float:
    if numerator_col not in df.columns or denominator_col not in df.columns:
        return np.nan
    den = df[denominator_col].sum()
    if den == 0:
        return np.nan
    return df[numerator_col].sum() / den

def chart_bar(df, x_col, y_col, title, xlabel=None, ylabel=None, rotation=0):
    ax = df.plot(kind="bar", x=x_col, y=y_col, figsize=(9, 5), legend=False)
    ax.set_title(title)
    ax.set_xlabel(xlabel or x_col)
    ax.set_ylabel(ylabel or y_col)
    plt.xticks(rotation=rotation)
    plt.tight_layout()
    plt.show()

def chart_line(df, x_col, y_col, title, xlabel=None, ylabel=None):
    ax = df.plot(kind="line", x=x_col, y=y_col, figsize=(11, 5), legend=False)
    ax.set_title(title)
    ax.set_xlabel(xlabel or x_col)
    ax.set_ylabel(ylabel or y_col)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 2. Load and Enrich Data

In [ ]:

files = {
    "advertisers": "advertisers.csv",
    "campaigns": "campaigns.csv",
    "ad_groups": "ad_groups.csv",
    "creatives": "creatives.csv",
    "placements": "placements.csv",
    "inventory": "inventory.csv",
    "markets": "markets.csv",
    "devices": "devices.csv",
    "daily_ad_performance": "daily_ad_performance.csv",
    "video_performance": "video_performance.csv",
    "conversion_events": "conversion_events.csv",
    "billing_revenue": "billing_revenue.csv",
    "data_quality_logs": "data_quality_logs.csv",
}

date_tables = {
    "daily_ad_performance",
    "video_performance",
    "conversion_events",
    "billing_revenue",
    "inventory",
    "data_quality_logs",
}

data = {
    table: read_csv_required(filename, parse_dates=["date"] if table in date_tables else None)
    for table, filename in files.items()
}

advertisers = data["advertisers"]
campaigns = data["campaigns"]
ad_groups = data["ad_groups"]
creatives = data["creatives"]
placements = data["placements"]
inventory = data["inventory"]
markets = data["markets"]
devices = data["devices"]
perf = data["daily_ad_performance"]
video = data["video_performance"]
conversions = data["conversion_events"]
billing = data["billing_revenue"]
dq_logs = data["data_quality_logs"]

print("Loaded tables:", len(data))


perf, split_date = add_period_by_date(perf, "date")
video, _ = add_period_by_date(video, "date")
inventory, _ = add_period_by_date(inventory, "date")
dq_logs, _ = add_period_by_date(dq_logs, "date")

perf_enriched = (
    perf
    .merge(advertisers, on="advertiser_id", how="left")
    .merge(campaigns, on="campaign_id", how="left", suffixes=("", "_campaign"))
    .merge(ad_groups, on="ad_group_id", how="left", suffixes=("", "_ad_group"))
    .merge(creatives, on="creative_id", how="left", suffixes=("", "_creative"))
    .merge(placements, on="placement_id", how="left", suffixes=("", "_placement"))
    .merge(markets, on="market_id", how="left", suffixes=("", "_market"))
    .merge(devices, on="device_id", how="left", suffixes=("", "_device"))
)

video_enriched = video.merge(
    perf_enriched[[
        col for col in [
            "performance_id",
            "creative_id",
            "period",
            "placement_id",
            "quality_tier",
            "platform",
            "market_name",
            "inventory_type",
            "advertiser_id",
            "advertiser_name",
            "video_duration_sec",
        ]
        if col in perf_enriched.columns
    ]].drop_duplicates(),
    on=[col for col in ["performance_id", "creative_id"] if col in video.columns and col in perf_enriched.columns],
    how="left",
    suffixes=("", "_perf"),
)

if "period_perf" in video_enriched.columns:
    video_enriched = video_enriched.drop(columns=["period_perf"])

print(f"Dynamic split date: {split_date.date()}")
print("Performance rows:", len(perf_enriched))
print("Video rows      :", len(video_enriched))


## 3. Helper Functions for RCA

In [ ]:

def summarize_dimension(df: pd.DataFrame, dimension_cols: list[str]) -> pd.DataFrame:
    summary = (
        df.groupby(["period", *dimension_cols], dropna=False, as_index=False)
        .agg(
            impressions=("impressions", "sum"),
            clicks=("clicks", "sum"),
            spend_usd=("spend_usd", "sum"),
            publisher_revenue_usd=("publisher_revenue_usd", "sum"),
            measurable_impressions=("measurable_impressions", "sum"),
            viewable_impressions=("viewable_impressions", "sum"),
        )
    )

    summary["impression_share"] = summary.groupby("period")["impressions"].transform(lambda x: x / x.sum())
    summary["revenue_share"] = summary.groupby("period")["publisher_revenue_usd"].transform(lambda x: x / x.sum())
    summary["viewability_rate"] = summary["viewable_impressions"] / summary["measurable_impressions"].replace(0, np.nan)
    summary["publisher_ecpm"] = summary["publisher_revenue_usd"] / summary["impressions"].replace(0, np.nan) * 1000

    return summary

def movement_table(summary: pd.DataFrame, dimension_cols: list[str]) -> pd.DataFrame:
    metrics = [
        "impressions",
        "impression_share",
        "publisher_revenue_usd",
        "revenue_share",
        "viewability_rate",
        "publisher_ecpm",
    ]

    base = summary[summary["period"] == "baseline"][dimension_cols + metrics].copy()
    decline = summary[summary["period"] == "decline"][dimension_cols + metrics].copy()

    if dimension_cols:
        out = base.merge(decline, on=dimension_cols, how="outer", suffixes=("_baseline", "_decline"))
    else:
        base["_join_key"] = 1
        decline["_join_key"] = 1
        out = base.merge(decline, on="_join_key", how="outer", suffixes=("_baseline", "_decline")).drop(columns="_join_key")

    for metric in metrics:
        out[f"{metric}_change_abs"] = out[f"{metric}_decline"] - out[f"{metric}_baseline"]
        out[f"{metric}_change_pct"] = out.apply(
            lambda x: pct_change(x[f"{metric}_decline"], x[f"{metric}_baseline"]),
            axis=1,
        )

    return out

def rca_rank(summary: pd.DataFrame, dimension_cols: list[str], dimension_name: str) -> pd.DataFrame:
    movement = movement_table(summary, dimension_cols)
    dim_label = movement[dimension_cols].astype(str).agg(" | ".join, axis=1)

    out = pd.DataFrame({
        "dimension": dimension_name,
        "segment": dim_label,
        "baseline_impression_share": movement["impression_share_baseline"],
        "decline_impression_share": movement["impression_share_decline"],
        "impression_share_change_abs": movement["impression_share_change_abs"],
        "baseline_viewability": movement["viewability_rate_baseline"],
        "decline_viewability": movement["viewability_rate_decline"],
        "viewability_change_abs": movement["viewability_rate_change_abs"],
        "decline_impressions": movement["impressions_decline"],
        "decline_revenue_usd": movement["publisher_revenue_usd_decline"],
    })

    # Portfolio-friendly heuristic:
    # high contribution = gained impression share and has below-average decline-period viewability.
    decline_avg_viewability = summary.loc[summary["period"] == "decline", "viewability_rate"].mean()
    out["below_decline_avg_viewability"] = decline_avg_viewability - out["decline_viewability"]
    out["rca_score"] = (
        out["impression_share_change_abs"].fillna(0).clip(lower=0)
        * out["below_decline_avg_viewability"].fillna(0).clip(lower=0)
    )

    return out.sort_values("rca_score", ascending=False)

def show_top_rca(rank_df: pd.DataFrame, n=10) -> pd.DataFrame:
    cols = [
        "dimension",
        "segment",
        "baseline_impression_share",
        "decline_impression_share",
        "impression_share_change_abs",
        "baseline_viewability",
        "decline_viewability",
        "viewability_change_abs",
        "rca_score",
    ]
    return rank_df[cols].head(n)


## 4. Executive Summary — What Happened?

In [ ]:

overall_summary = summarize_dimension(perf_enriched, [])
overall_movement = movement_table(overall_summary, [])

overall_summary


In [ ]:

overall_comparison = pd.DataFrame([
    {
        "metric": "impressions",
        "baseline": overall_summary.loc[overall_summary["period"] == "baseline", "impressions"].iloc[0],
        "decline": overall_summary.loc[overall_summary["period"] == "decline", "impressions"].iloc[0],
    },
    {
        "metric": "publisher_revenue_usd",
        "baseline": overall_summary.loc[overall_summary["period"] == "baseline", "publisher_revenue_usd"].iloc[0],
        "decline": overall_summary.loc[overall_summary["period"] == "decline", "publisher_revenue_usd"].iloc[0],
    },
    {
        "metric": "viewability_rate",
        "baseline": overall_summary.loc[overall_summary["period"] == "baseline", "viewability_rate"].iloc[0],
        "decline": overall_summary.loc[overall_summary["period"] == "decline", "viewability_rate"].iloc[0],
    },
    {
        "metric": "publisher_ecpm",
        "baseline": overall_summary.loc[overall_summary["period"] == "baseline", "publisher_ecpm"].iloc[0],
        "decline": overall_summary.loc[overall_summary["period"] == "decline", "publisher_ecpm"].iloc[0],
    },
])

overall_comparison["change_pct"] = overall_comparison.apply(lambda x: pct_change(x["decline"], x["baseline"]), axis=1)
overall_comparison["change_pct_fmt"] = overall_comparison["change_pct"].map(fmt_pct)
overall_comparison


## 5. RCA by Placement Quality Tier

In [ ]:

quality_summary = summarize_dimension(perf_enriched, ["quality_tier"])
quality_rank = rca_rank(quality_summary, ["quality_tier"], "Placement Quality Tier")

display(quality_summary.sort_values(["period", "impression_share"], ascending=[True, False]))
show_top_rca(quality_rank)


In [ ]:

quality_chart = quality_summary.pivot(index="quality_tier", columns="period", values="impression_share")
quality_chart.plot(kind="bar", figsize=(9, 5))
plt.title("Impression Share by Placement Quality Tier")
plt.xlabel("Quality Tier")
plt.ylabel("Impression Share")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 6. RCA by Placement

In [ ]:

placement_cols = [col for col in ["placement_id", "placement_name", "quality_tier", "inventory_type"] if col in perf_enriched.columns]
placement_summary = summarize_dimension(perf_enriched, placement_cols)
placement_rank = rca_rank(placement_summary, placement_cols, "Placement")

show_top_rca(placement_rank, n=15)


## 7. RCA by Device / Platform

In [ ]:

platform_cols = [col for col in ["platform"] if col in perf_enriched.columns]
platform_summary = summarize_dimension(perf_enriched, platform_cols)
platform_rank = rca_rank(platform_summary, platform_cols, "Device / Platform")

display(platform_summary.sort_values(["period", "impression_share"], ascending=[True, False]))
show_top_rca(platform_rank)


In [ ]:

platform_chart = platform_summary.pivot(index="platform", columns="period", values="viewability_rate")
platform_chart.plot(kind="bar", figsize=(9, 5))
plt.title("Viewability by Platform")
plt.xlabel("Platform")
plt.ylabel("Viewability Rate")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 8. RCA by Market

In [ ]:

market_cols = [col for col in ["market_name", "region", "market_maturity"] if col in perf_enriched.columns]
market_summary = summarize_dimension(perf_enriched, market_cols)
market_rank = rca_rank(market_summary, market_cols, "Market")

show_top_rca(market_rank, n=15)


## 9. RCA by Inventory Type

In [ ]:

inventory_cols = [col for col in ["inventory_type"] if col in perf_enriched.columns]
inventory_type_summary = summarize_dimension(perf_enriched, inventory_cols)
inventory_type_rank = rca_rank(inventory_type_summary, inventory_cols, "Inventory Type")

display(inventory_type_summary.sort_values(["period", "impression_share"], ascending=[True, False]))
show_top_rca(inventory_type_rank)


## 10. Creative Duration and VCR RCA

In [ ]:

if "video_duration_sec" not in video_enriched.columns:
    raise KeyError("video_duration_sec column is required for creative duration RCA.")

video_enriched["duration_bucket"] = pd.cut(
    video_enriched["video_duration_sec"],
    bins=[0, 10, 15, 30, np.inf],
    labels=["<=10s", "11-15s", "16-30s", ">30s"],
    include_lowest=True,
)

duration_summary = (
    video_enriched.groupby(["period", "duration_bucket"], observed=False, as_index=False)
    .agg(
        video_starts=("video_starts", "sum"),
        video_completes=("video_completes", "sum"),
    )
)

duration_summary["vcr"] = duration_summary["video_completes"] / duration_summary["video_starts"].replace(0, np.nan)
duration_summary["start_share"] = duration_summary.groupby("period")["video_starts"].transform(lambda x: x / x.sum())

duration_summary


In [ ]:

duration_chart = duration_summary.pivot(index="duration_bucket", columns="period", values="vcr")
duration_chart.plot(kind="bar", figsize=(9, 5))
plt.title("VCR by Creative Duration Bucket")
plt.xlabel("Duration Bucket")
plt.ylabel("Video Completion Rate")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 11. Advertiser Mix RCA

In [ ]:

advertiser_cols = [col for col in ["advertiser_id", "advertiser_name", "industry", "advertiser_tier"] if col in perf_enriched.columns]
advertiser_summary = summarize_dimension(perf_enriched, advertiser_cols)
advertiser_rank = rca_rank(advertiser_summary, advertiser_cols, "Advertiser Mix")

show_top_rca(advertiser_rank, n=15)


## 12. Data Quality Assessment

In [ ]:

dq_group_cols = [col for col in ["period", "issue_type", "severity"] if col in dq_logs.columns]

if len(dq_group_cols) > 1:
    dq_summary = (
        dq_logs.groupby(dq_group_cols, dropna=False, as_index=False)
        .agg(
            logs=("dq_log_id", "count") if "dq_log_id" in dq_logs.columns else ("date", "count"),
            estimated_affected_rows=("estimated_affected_rows", "sum") if "estimated_affected_rows" in dq_logs.columns else ("date", "count"),
            root_cause_candidates=("is_root_cause_candidate", "sum") if "is_root_cause_candidate" in dq_logs.columns else ("date", "count"),
        )
    )
else:
    dq_summary = pd.DataFrame()

total_perf_rows = len(perf_enriched)
dq_affected_rows = dq_logs["estimated_affected_rows"].sum() if "estimated_affected_rows" in dq_logs.columns else np.nan
dq_affected_ratio = dq_affected_rows / total_perf_rows if total_perf_rows else np.nan

display(dq_summary)
print(f"Total performance rows        : {total_perf_rows:,}")
print(f"Estimated DQ affected rows    : {dq_affected_rows:,.0f}" if pd.notna(dq_affected_rows) else "Estimated DQ affected rows    : n/a")
print(f"Estimated affected row ratio  : {fmt_pct(dq_affected_ratio)}")


## 13. Ranked RCA Contribution Summary

In [ ]:

rca_tables = [
    quality_rank,
    placement_rank,
    platform_rank,
    market_rank,
    inventory_type_rank,
    advertiser_rank,
]

combined_rca_rank = (
    pd.concat(rca_tables, ignore_index=True)
    .sort_values("rca_score", ascending=False)
    .reset_index(drop=True)
)

show_top_rca(combined_rca_rank, n=20)


## 14. Recommended Business Actions

In [ ]:

top_quality = show_top_rca(quality_rank, 1)
top_platform = show_top_rca(platform_rank, 1)
top_inventory = show_top_rca(inventory_type_rank, 1)

actions = pd.DataFrame([
    {
        "priority": "P0",
        "action": "Introduce placement quality guardrail",
        "rationale": "Reduce delivery pressure into segments with rising impression share and weaker viewability.",
        "owner": "Ads Marketplace / Inventory Quality",
    },
    {
        "priority": "P0",
        "action": "Add mobile web-specific monitoring",
        "rationale": "Mobile web appears as a key segment to monitor when impression mix shifts and quality drops.",
        "owner": "Ads Product / Measurement",
    },
    {
        "priority": "P1",
        "action": "Create advertiser-facing quality diagnostics",
        "rationale": "Help commercial teams explain why revenue can stay stable while quality metrics decline.",
        "owner": "Analytics / Commercial Strategy",
    },
    {
        "priority": "P1",
        "action": "Recommend shorter video creative variants",
        "rationale": "Longer creatives tend to show lower VCR and should be tested against shorter versions.",
        "owner": "Creative Strategy / Advertiser Success",
    },
    {
        "priority": "P2",
        "action": "Keep DQ monitor as supporting evidence",
        "rationale": "DQ issues exist but should be treated as a secondary check unless affected volume grows materially.",
        "owner": "Data Engineering / Analytics Engineering",
    },
])

actions


## 15. Portfolio RCA Narrative

In [ ]:

revenue_change = overall_comparison.loc[overall_comparison["metric"] == "publisher_revenue_usd", "change_pct"].iloc[0]
impression_change = overall_comparison.loc[overall_comparison["metric"] == "impressions", "change_pct"].iloc[0]
viewability_change = overall_comparison.loc[overall_comparison["metric"] == "viewability_rate", "change_pct"].iloc[0]

top_segments = show_top_rca(combined_rca_rank, n=5)

narrative = f"""
WHAT HAPPENED
Revenue stayed broadly stable ({fmt_pct(revenue_change)}), while impressions increased ({fmt_pct(impression_change)}).
However, viewability declined materially ({fmt_pct(viewability_change)}), indicating a quality deterioration rather than a pure monetization issue.

WHERE IT HAPPENED
The highest-ranked RCA segments are shown in the combined RCA table. These are segments that gained impression share during the decline period and had weaker viewability.

WHY IT LIKELY HAPPENED
The evidence points to a delivery mix shift toward lower-quality inventory, especially when combined with device/platform and placement-level patterns.
For video, longer creative duration also contributes to lower completion rate.

WHY DATA QUALITY IS NOT THE MAIN CAUSE
Data quality logs exist, but the estimated affected row ratio is {fmt_pct(dq_affected_ratio)} of daily performance rows.
This makes DQ a monitoring concern, but not the primary explanation for a broad quality metric decline.

RECOMMENDED ACTION
Prioritize placement quality guardrails, mobile web monitoring, and advertiser-facing quality diagnostics.
"""

print(narrative)
display(top_segments)
